In [1]:
import kagglehub
import os
import sys
import pandas as pd
import logging
import librosa
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, Dense, Dropout
import warnings
warnings.filterwarnings("ignore")

# --- 1. SET UP LOGGING ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

c:\Users\ASUS\Desktop\AI-Use cases\MindMosaic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 2. DOWNLOAD THE RAVDESS DATASET ---
# This part of the code handles downloading the dataset from Kaggle.
logging.info("Downloading RAVDESS dataset from Kaggle...")
try:
    # This downloads the dataset and returns the path to the extracted files.
    # The dataset contains a main directory 'audio_speech_actors_01-24'.
    path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
    dataset_path = os.path.join(path, "audio_speech_actors_01-24")
    logging.info(f"Dataset downloaded to: {dataset_path}")

    # Check if the expected directory exists.
    if not os.path.isdir(dataset_path):
        raise FileNotFoundError(f"Could not find dataset directory at {dataset_path}")
except Exception as e:
    logging.error(f"Failed to download dataset from Kaggle: {e}")
    sys.exit(1)

2025-08-14 21:03:04,462 - INFO - Downloading RAVDESS dataset from Kaggle...
2025-08-14 21:03:10,256 - INFO - Dataset downloaded to: C:\Users\ASUS\.cache\kagglehub\datasets\uwrfkaggler\ravdess-emotional-speech-audio\versions\1\audio_speech_actors_01-24


In [3]:
# --- 3. PROCESS THE DOWNLOADED DATA ---
# This part of the code is adapted from your original snippet.
# It now uses the `dataset_path` variable from the download step.

# Get the list of actor directories from the downloaded dataset path.
try:
    ravdess_directory_list = os.listdir(dataset_path)
    audio_paths = []
    labels = []

    for actor_dir in ravdess_directory_list:
        # Check if the item is a directory before trying to list its contents.
        actor_full_path = os.path.join(dataset_path, actor_dir)
        if not os.path.isdir(actor_full_path):
            continue

        for file in os.listdir(actor_full_path):
            part = file.split('.')[0].split('-')
            
            # The emotion code is the third part (index 2) of the filename.
            # Convert to an integer as per the mapping.
            emotion_code = int(part[2])
            labels.append(emotion_code)
            
            # Store the full file path.
            audio_paths.append(os.path.join(actor_full_path, file))

    # --- 4. CREATE PANDAS DATAFRAME ---
    # Creating the DataFrame directly with the desired column names.
    Ravdess_df = pd.DataFrame({"path": audio_paths, "label": labels})

    # Replace the numerical emotion codes with human-readable labels.
    Ravdess_df.label.replace({
        1:'neutral', 2:'calm', 3:'happy', 4:'sad', 5:'angry', 6:'fear', 7:'disgust', 8:'surprise'
    }, inplace=True)

    # Print the first 5 rows to verify the result.
    print("\nDataFrame created successfully:")
    print(Ravdess_df.head())

except Exception as e:
    logging.error(f"Failed to process dataset files: {e}")
    sys.exit(1)


except Exception as e:
    logging.error(f"Failed to process dataset files: {e}")
    sys.exit(1)



DataFrame created successfully:
                                                path    label
0  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
1  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
2  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
3  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
4  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...     calm


In [4]:
Ravdess_df['label'].value_counts()

label
calm        192
happy       192
sad         192
angry       192
disgust     192
fear        192
surprise    192
neutral      96
Name: count, dtype: int64

In [5]:
import numpy as np
def extract_mfcc_sequence(file_path, max_len=130):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc = mfcc.T
    if mfcc.shape[0] < max_len:
        pad_width = max_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, pad_width=((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:max_len]
    return mfcc

In [6]:
X = np.array([extract_mfcc_sequence(p) for p in Ravdess_df['path']])
y = np.array(Ravdess_df['label'])


In [7]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_encoded = tf.keras.utils.to_categorical(y_encoded)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [8]:
model = Sequential([
    Conv1D(64, 5, activation='relu', input_shape=(130, 40)),
    Dropout(0.3),
    Conv1D(64, 3, activation='relu'),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(8, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 126, 64)        │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 126, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 124, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,896 (116.78 KB)

 Trainable params: 29,896 (116.78 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=64)

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9462 - loss: 0.1478 - val_accuracy: 0.6875 - val_loss: 1.1469
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9557 - loss: 0.1392 - val_accuracy: 0.7083 - val_loss: 1.2092
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9618 - loss: 0.1293 - val_accuracy: 0.6910 - val_loss: 1.2036
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9601 - loss: 0.1448 - val_accuracy: 0.6736 - val_loss: 1.3038
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9549 - loss: 0.1416 - val_accuracy: 0.7083 - val_loss: 1.2379
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9679 - loss: 0.1107 - val_accuracy: 0.7049 - val_loss: 1.1387
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9679 - loss: 0.1112 - val_accuracy: 0.7292 - val_loss: 1.2003
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9644 - loss: 0.1130 - val_accuracy: 0.

In [13]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7396 - loss: 1.4335 
Test Accuracy: 0.7395833134651184


In [ ]:
model.save("ravdess_audio_model_final.hdf5")
logging.info("Model saved successfully as 'ravdess_audio_model_final.hdf5'")


2025-08-14 21:10:37,329 - WARNING - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 
2025-08-14 21:10:37,375 - INFO - Model saved successfully as 'ravdess_audio_model_final.hdf5'
